# RAG Engine - GPU Testing with Google Colab

This notebook tests the RAG Inference Engine on a T4 GPU.

**Features tested:**
- CUDA-accelerated embedding (ONNX Runtime)
- Query batching (5-8x throughput improvement)
- FAISS HNSW vector search
- HTTP API endpoints
- Latency benchmarks

## 1. Setup Environment

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Verify CUDA version
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Install system dependencies
!apt-get update
!apt-get install -y cmake build-essential git curl wget unzip libprotobuf-dev protobuf-compiler libuv1-dev

In [ ]:
# Clone repository
!rm -rf rag-engine
!git clone https://github.com/gumeeee/rag-engine.git
%cd rag-engine
!ls -la

## 2. Install vcpkg and Dependencies

In [ ]:
# Install vcpkg
%cd /content
!rm -rf vcpkg
!git clone https://github.com/Microsoft/vcpkg.git
!./vcpkg/bootstrap-vcpkg.sh

In [ ]:
# Install C++ dependencies via vcpkg (this may take a while)
!export VCPKG_ROOT=/content/vcpkg
!/content/vcpkg/vcpkg install faiss-cpu onnxruntime-cuda libuv protobuf spdlog nlohmann-json --overlay-triplets=/content/rag-engine/triplets

## 3. Download ONNX Model

In [ ]:
# Install Python dependencies
!pip install sentence-transformers torch numpy requests -q

In [ ]:
# Download and convert model to ONNX
import os
os.makedirs('/content/rag-engine/models', exist_ok=True)

from sentence_transformers import SentenceTransformer
import torch

print('Loading sentence-transformers model...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Model loaded successfully!')

# Save as PyTorch (for reference)
model.save('/content/rag-engine/models/all-MiniLM-L6-v2')
print('Model saved to models/all-MiniLM-L6-v2')

## 4. Build RAG Engine

In [ ]:
# Build with CMake
%cd /content/rag-engine
!mkdir -p build && cd build
!cmake .. \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_TOOLCHAIN_FILE=/content/vcpkg/scripts/buildsystems/vcpkg.cmake \
  -DVCPKG_TARGET_TRIPLET=x64-linux.Release \
  -Donnxruntime_DIR=/content/vcpkg/installed/x64-linux/share/onnxruntime-cuda
!cmake --build . --config Release -j$(nproc)

## 5. Generate Test Corpus

In [ ]:
# Create sample corpus
import os

os.makedirs('/content/rag-engine/data/corpus', exist_ok=True)

documents = {
    'attention_is_all_you_need.txt': '''
    Attention is all you need. The Transformer architecture was introduced in 2017
    and has become the foundation for many state-of-the-art models including BERT,
    GPT, and T5. It uses self-attention mechanisms to process input sequences
    in parallel, allowing for faster training and better handling of long-range
    dependencies compared to RNNs.
    ''',
    'bert.txt': '''
    BERT uses bidirectional transformers for language understanding. It was
    pre-trained on a large corpus using masked language modeling and next
    sentence prediction objectives. This allows BERT to capture context from
    both directions, leading to state-of-the-art results on many NLP tasks.
    ''',
    'gpt.txt': '''
    GPT models are autoregressive transformers trained on large amounts of
    text data. They use next-token prediction during training and can generate
    coherent text given a prompt. GPT-3 has 175 billion parameters and can
    perform many tasks without task-specific training.
    ''',
    'rag_intro.txt': '''
    Retrieval-Augmented Generation combines the power of large language models
    with external knowledge retrieval. This allows models to access up-to-date
    information and reduces hallucination. RAG is particularly useful for
    question answering over private or specialized documents.
    ''',
    'vector_search.txt': '''
    Vector similarity search uses embeddings to find semantically similar items.
    Approximate Nearest Neighbor algorithms like HNSW enable fast search over
    millions of vectors. FAISS is a popular library for efficient similarity
    search and clustering of dense vectors.
    '''
}

for filename, content in documents.items():
    with open(f'/content/rag-engine/data/corpus/{filename}', 'w') as f:
        f.write(content.strip())

print(f'Created {len(documents)} sample documents')

In [ ]:
# Generate embeddings
%cd /content/rag-engine
!python scripts/generate_embeddings.py \
  --corpus-dir ./data/corpus \
  --output ./data/corpus.corpus \
  --model sentence-transformers/all-MiniLM-L6-v2

## 6. Run Unit Tests

In [ ]:
# Build and run tests
%cd /content/rag-engine/build
!cmake --build . --target rag-engine-tests -j$(nproc)
!./rag-engine-tests

## 7. Start Server and Test API

In [ ]:
# Start server in background
import subprocess
import time

# Set environment variables
import os
os.environ['RAG_CONFIG_PATH'] = '/content/rag-engine/config/default_config.pb.txt'
os.environ['RAG_CORPUS_PATH'] = '/content/rag-engine/data/corpus.corpus'
os.environ['RAG_LOG_LEVEL'] = 'info'

# Start server
server_proc = subprocess.Popen(
    ['/content/rag-engine/build/rag-engine', '--config', '/content/rag-engine/config/default_config.pb.txt'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print('Server starting...')
time.sleep(5)
print('Server should be running on port 8080')

In [ ]:
# Test health endpoint
import requests

try:
    response = requests.get('http://localhost:8080/health', timeout=5)
    print(f'Health check: {response.json()}')
except Exception as e:
    print(f'Health check failed: {e}')
    print('Server may not be ready yet, trying again...')
    time.sleep(5)
    response = requests.get('http://localhost:8080/health', timeout=5)
    print(f'Health check: {response.json()}')

In [ ]:
# Test query endpoint
import requests
import time

queries = [
    'What is the Transformer architecture?',
    'How does BERT work?',
    'What is RAG?',
    'How does vector search work?',
]

for query in queries:
    start = time.time()
    response = requests.post(
        'http://localhost:8080/query',
        json={'query_text': query, 'top_k': 3},
        timeout=30
    )
    elapsed = (time.time() - start) * 1000
    
    if response.ok:
        result = response.json()
        print(f'Query: {query}')
        print(f'  Latency: {elapsed:.1f}ms')
        print(f'  Results: {len(result.get("results", []))}')
        print()
    else:
        print(f'Error: {response.status_code} - {response.text}')

## 8. Run Benchmarks

In [ ]:
# Run latency benchmark
%cd /content/rag-engine
!python scripts/benchmark.py \
  --endpoint http://localhost:8080 \
  --queries 50 \
  --concurrency 10

In [ ]:
# Run batching benchmark
%cd /content/rag-engine
!python scripts/benchmark_batching.py \
  --num-queries 100

## 9. Cleanup

In [ ]:
# Stop server
server_proc.terminate()
server_proc.wait()
print('Server stopped')

# Print summary
print('\n' + '='*50)
print('BENCHMARK COMPLETE')
print('='*50)
print('\nGPU Tested: NVIDIA T4')
print('Model: sentence-transformers/all-MiniLM-L6-v2')
print('Index: FAISS HNSW')
print('\nCheck the outputs above for latency results.')